---
# Clinical Data Quality Engine (DQIE)
# Notebook 02 — Validation Layer
# Purpose: Run structural, relational, temporal, business-rule, and OCR validations
---

# 1. Setup

In [ ]:
import sys
import os
import pandas as pd
from pathlib import Path

# Add project root to PYTHONPATH
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PYTHONPATH OK:", PROJECT_ROOT)

In [ ]:
from src._3_validation.validate_schema import SchemaValidator
from src._3_validation.validate_csv import CSVValidator
from src._3_validation.validate_sql import SQLValidator
from src._3_validation.validate_relations import RelationsValidator
from src._3_validation.validate_business_rules import BusinessRulesValidator
from src._3_validation.validate_ocr import OCRValidator

SILVER_DIR = Path("../data/_2_silver")
print("Silver directory:", SILVER_DIR)

In [ ]:
df_patients = pd.read_parquet(SILVER_DIR / "patients.parquet")
df_injuries = pd.read_parquet(SILVER_DIR / "injuries.parquet")
df_sessions = pd.read_parquet(SILVER_DIR / "sessions.parquet")
df_clinical = pd.read_parquet(SILVER_DIR / "clinical_reports.parquet")
df_ocr_json = pd.read_parquet(SILVER_DIR / "ocr_extracted.parquet")
df_ocr_images = pd.read_parquet(SILVER_DIR / "ocr_images.parquet")

print("Loaded Silver datasets.")


# 2. Schema validation


In [ ]:
schema_validator = SchemaValidator()

print("\n## Schema — patients")
schema_validator.validate(
    df_patients,
    [
        "patient_id","first_name","last_name","age","sex","injury_type",
        "diagnosis_code","diagnosis_category","start_date","end_date"
    ],
    "patients"
)

In [ ]:
print("\n## Schema — injuries")
schema_validator.validate(
    df_injuries,
    [
        "injury_type","typical_recovery_days","typical_sessions",
        "typical_pain_initial","typical_mobility_initial"
    ],
    "injuries"
)

In [ ]:
print("\n## Schema — sessions")
schema_validator.validate(
    df_sessions,
    [
        "session_id","patient_id","session_date","session_number","injury_type",
        "pain_initial","pain_final","mobility_initial","mobility_final",
        "therapist_id","therapist_notes","recovery_days"
    ],
    "sessions"
)

In [ ]:
print("\n## Schema — clinical_reports")
schema_validator.validate(
    df_clinical,
    [
        "report_id","patient_id","report_date","diagnosis_text",
        "pain_score","mobility_score","notes"
    ],
    "clinical_reports"
)

In [ ]:
print("\n## Schema — ocr_extracted")
schema_validator.validate(
    df_ocr_json,
    [
        "ocr_id","patient_id","report_date","extracted_text",
        "extracted_pain","extracted_mobility","image_path"
    ],
    "ocr_extracted"
)

In [ ]:
print("\n## Schema — ocr_images")
schema_validator.validate(
    df_ocr_images,
    [
        "image_file","image_path","extracted_text"
    ],
    "ocr_images"
)


# 3. CSV & SQL validation


In [ ]:
csv_validator = CSVValidator()
sql_validator = SQLValidator()

print("\n## CSV Validation — patients")
csv_validator.validate(df_patients, "patients")

In [ ]:
print("\n## CSV Validation — sessions")
csv_validator.validate(df_sessions, "sessions")

In [ ]:
print("\n## SQL Validation — patients")
sql_validator.validate(df_patients, "patients")

In [ ]:
print("\n## SQL Validation — sessions")
sql_validator.validate(df_sessions, "sessions")


# 4. Referential integrity validation


In [ ]:
relations_validator = RelationsValidator()

print("\n## Relations — patients, injuries, sessions, clinical_reports")
relation_errors = relations_validator.validate(
    patients_df=df_patients,
    injuries_df=df_injuries,
    sessions_df=df_sessions,
    reports_df=df_clinical
)

In [ ]:
if relation_errors:
    print("Relations validation found issues:")
    for err in relation_errors:
        print(" -", err)
else:
    print("Relations validation passed with no errors.")


# 5. Business rules validation


In [ ]:
business_validator = BusinessRulesValidator()

print("\n## Business Rules — sessions")
business_errors = business_validator.validate(df_sessions)

In [ ]:
if business_errors:
    print("Business rules validation found issues:")
    for err in business_errors:
        print(" -", err)
else:
    print("Business rules validation passed with no errors.")


# 6. OCR-specific validation


In [ ]:
ocr_validator = OCRValidator()

ocr_validator = OCRValidator()

print("\n## OCR Validation — JSON OCR")
ocr_json_errors = ocr_validator.validate(
    df=df_ocr_json,
    df_name="ocr_extracted.json",
    text_col="extracted_text",
    required_cols=["patient_id", "extracted_text", "report_date", "image_path"],
    min_len=20,
    keywords=["pain", "mobility", "session", "treatment"]
)

In [ ]:
if ocr_json_errors:
    print("OCR JSON validation found issues:")
    for err in ocr_json_errors:
        print(" -", err)
else:
    print("OCR JSON validation passed with no errors.")

In [ ]:
print("\n## OCR Validation — Image OCR")
ocr_image_errors = ocr_validator.validate(
    df=df_ocr_images,
    df_name="ocr_images",
    text_col="extracted_text",
    required_cols=["image_file", "image_path", "extracted_text"],
    min_len=10,
    noise_chars=["#", "@", "%", "&", "*"],
    max_noise_ratio=0.15
)

In [ ]:
if ocr_image_errors:
    print("OCR image validation found issues:")
    for err in ocr_image_errors:
        print(" -", err)
else:
    print("OCR image validation passed with no errors.")


# 7. Summary

In [ ]:
print("\n--- VALIDATION SUMMARY ---")
print("Schema validation completed.")
print("CSV & SQL validation completed.")
print("Referential integrity validation completed.")
print("Business rules validation completed.")
print("OCR validation completed.")
print("Silver layer is validated and ready for anomaly detection.")
